# Tetranet 10M: Full Training + Analysis
---
**Datasets needed:**
- `tinystories-full` (TinyStoriesV2-GPT4-train.txt + TinyStoriesV2-GPT4-valid.txt)  
- `tetranet-code` (train_kaggle.py, model.py, quaternary.py, regularization.py, eval_all.py, tetranet_tokenizer.json)

**Run order:** Cell 0 → **Runtime → Restart and run all**

In [ ]:
# Cell 0: Fix P100 incompatibility (run once, then Restart & Run All)
!pip install -q torch==2.5.1 --index-url https://download.pytorch.org/whl/cu124
!pip install -q tokenizers matplotlib pandas seaborn

In [ ]:
# Cell 1: Setup - copy code + data from Kaggle Datasets
import json, math, os, shutil, subprocess, sys, time
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams.update({
    "figure.dpi": 150, "font.size": 11, "axes.titlesize": 13,
    "axes.labelsize": 11, "legend.fontsize": 9,
    "xtick.labelsize": 9, "ytick.labelsize": 9,
})

os.makedirs("/kaggle/working/tetranet", exist_ok=True)
for f in ["train_kaggle.py", "model.py", "quaternary.py", "regularization.py",
          "eval_all.py", "tetranet_tokenizer.json"]:
    shutil.copy(f"/kaggle/input/tetranet-code/{f}", f"/kaggle/working/tetranet/{f}")
os.chdir("/kaggle/working/tetranet")

os.makedirs("./tinystories", exist_ok=True)
for split in ["train", "valid"]:
    shutil.copy(
        f"/kaggle/input/tinystories-full/TinyStoriesV2-GPT4-{split}.txt",
        f"./tinystories/TinyStories-{split}.txt")

DEVICE = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f"GPU: {DEVICE}")
print(f"VRAM: {VRAM:.1f} GB")

In [ ]:
# Cell 2: Train all 6 baselines at 10M scale
BASELINES = [
    ("full_precision", ""),
    ("bitnet", ""),
    ("uniform_2bit", ""),
    ("fixed_c_025", ""),
    ("fixed_c_05", ""),
    ("learned_c", "--c-lr 0.003 --alpha 2.0 --snap-start 0.4"),
]

SMALL_CONFIG = "--hidden-dim 256 --num-layers 8 --num-heads 8 --ffn-dim 1024"

for name, extra_args in BASELINES:
    ckpt = f"./checkpoint_{name}.pt"
    log = f"./log_{name}.csv"
    print(f"\n{'='*60}")
    print(f"Training: {name}")
    print(f"{'='*60}")
    cmd = (
        f"python train_kaggle.py --baseline {name} "
        f"--data-path ./tinystories/TinyStories-train.txt "
        f"--tokenizer-path ./tetranet_tokenizer.json "
        f"--max-stories 100000 {SMALL_CONFIG} "
        f"--batch-size 8 --grad-accum 4 "
        f"--lr 3e-4 --weight-decay 0.1 "
        f"--epochs 1 "
        f"--checkpoint-path {ckpt} --log-path {log} "
        f"--ckpt-interval 5000 --log-interval 50 "
        f"--bf16 {extra_args}"
    )
    t0 = time.time()
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    elapsed = (time.time() - t0) / 60
    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr[:500])
    print(f"  -> {name} finished in {elapsed:.1f} min")

print("\n All baselines trained.")

In [ ]:
# Cell 3: Load all logs into a dataframe
BASELINE_NAMES = ["full_precision", "bitnet", "uniform_2bit",
                  "fixed_c_025", "fixed_c_05", "learned_c"]
COLORS = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2", "#937860"]
LABELS = ["Full precision", "BitNet b1.58", "Uniform 2-bit",
          "Quaternary c=0.25", "Quaternary c=0.5", "Quaternary learned_c"]

logs = {}
for name in BASELINE_NAMES:
    df = pd.read_csv(f"./log_{name}.csv")
    if "c_values_json" in df.columns:
        df["c_values"] = df["c_values_json"].apply(
            lambda x: json.loads(x) if isinstance(x, str) else {})
    logs[name] = df
    print(f"{name:20s}  {len(df):>4d} rows  "
          f"final loss={df['task_loss'].iloc[-1]:.4f}  "
          f"best loss={df['task_loss'].min():.4f}")

In [ ]:
# Cell 4: Fig 1 - Training loss overlay (all baselines)
fig, ax = plt.subplots(figsize=(10, 5))
for i, name in enumerate(BASELINE_NAMES):
    df = logs[name]
    ax.plot(df["step"], df["task_loss"], color=COLORS[i], label=LABELS[i], linewidth=1.5)
ax.set_xlabel("Step")
ax.set_ylabel("Cross-entropy loss")
ax.set_title("Training loss curves - 10M models on TinyStories")
ax.legend(framealpha=0.9)
ax.set_xlim(left=0)
fig.tight_layout()
fig.savefig("fig1_training_loss.png", dpi=150)
plt.show()

In [ ]:
# Cell 5: Fig 2 - Gradient norm comparison
fig, ax = plt.subplots(figsize=(10, 5))
for i, name in enumerate(BASELINE_NAMES):
    df = logs[name]
    if "grad_norm" in df.columns:
        ax.plot(df["step"], df["grad_norm"], color=COLORS[i],
                label=LABELS[i], linewidth=1, alpha=0.8)
ax.set_xlabel("Step")
ax.set_ylabel("Gradient norm")
ax.set_title("Gradient norm during training")
ax.legend(framealpha=0.9, ncol=2)
ax.set_yscale("log")
fig.tight_layout()
fig.savefig("fig2_grad_norm.png", dpi=150)
plt.show()

In [ ]:
# Cell 6: Fig 3 - PPL comparison bar chart
from models import build_model
from eval_all import eval_ppl, load_tokenizer, tokenize_valid, ValidDataset

SEQ_LEN, BATCH_SIZE, MAX_STORIES = 512, 4, 2500
tokenizer = load_tokenizer("./tetranet_tokenizer.json")
tokens = tokenize_valid("./tinystories/TinyStories-valid.txt", tokenizer, MAX_STORIES)
dataset = ValidDataset(tokens, SEQ_LEN)
loader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
print(f"Validation set: {len(dataset):,} sequences\n")

ppl_results = {}
for name in BASELINE_NAMES:
    ckpt_path = f"./checkpoint_{name}_final.pt"
    if not os.path.exists(ckpt_path):
        print(f"SKIP {name}: {ckpt_path} not found"); continue
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    model = build_model(baseline=name, vocab_size=4096, hidden_dim=256,
                        num_layers=8, num_heads=8, ffn_dim=1024)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    loss, ppl = eval_ppl(model, loader)
    ppl_results[name] = {"loss": loss, "ppl": ppl}
    print(f"{name:20s}  loss={loss:.4f}  ppl={ppl:.2f}")

fig, ax = plt.subplots(figsize=(10, 5))
vals = [ppl_results[n]["ppl"] for n in BASELINE_NAMES if n in ppl_results]
labels_plot = [LABELS[i] for i, n in enumerate(BASELINE_NAMES) if n in ppl_results]
colors_plot = [COLORS[i] for i, n in enumerate(BASELINE_NAMES) if n in ppl_results]
bars = ax.bar(labels_plot, vals, color=colors_plot, width=0.6, edgecolor="white")
ax.set_ylabel("Perplexity (lower is better)")
ax.set_title("Perplexity comparison - TinyStories validation (10M models)")
ax.tick_params(axis="x", rotation=20)

for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
            f"{val:.2f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

if "full_precision" in ppl_results:
    fp_ppl = ppl_results["full_precision"]["ppl"]
    for bar, val in zip(bars, vals):
        delta = val - fp_ppl
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()/2,
                f"{delta:+.2f} vs FP", ha="center", va="center",
                fontsize=7, color="white" if np.mean(bar.get_facecolor()[:3]) < 0.5 else "black",
                fontweight="bold")

fig.tight_layout()
fig.savefig("fig3_ppl_comparison.png", dpi=150)
plt.show()

In [ ]:
# Cell 7: Fig 4 - Structural fingerprinting
ckpt_lc = torch.load("./checkpoint_learned_c_final.pt", map_location="cpu", weights_only=False)
model_lc = build_model(baseline="learned_c", vocab_size=4096, hidden_dim=256,
                       num_layers=8, num_heads=8, ffn_dim=1024)
model_lc.load_state_dict(ckpt_lc["model_state_dict"])
c_vals = model_lc.get_c_values()

proj_map = {"q_proj": "Q", "k_proj": "K", "v_proj": "V", "o_proj": "O",
            "gate_proj": "Gate", "up_proj": "Up", "down_proj": "Down"}
by_type = {}
for key, val in c_vals.items():
    for pat, label in proj_map.items():
        if pat in key:
            by_type.setdefault(label, []).append(val); break

types = sorted(by_type.keys())
x = np.arange(len(types))
counts_025 = [sum(1 for v in by_type[t] if abs(v - 0.25) < 0.02) for t in types]
counts_05  = [sum(1 for v in by_type[t] if abs(v - 0.50) < 0.02) for t in types]

fig, ax = plt.subplots(figsize=(9, 4.5))
w = 0.35
b1 = ax.bar(x - w/2, counts_025, w, label="Snapped to 0.25",
            color="#4C72B0", edgecolor="white")
b2 = ax.bar(x + w/2, counts_05,  w, label="Snapped to 0.50",
            color="#DD8452", edgecolor="white")
ax.set_ylabel("Number of layers (out of 8)")
ax.set_title("Structural fingerprinting: per-projection c snapping")
ax.set_xticks(x); ax.set_xticklabels(types)
ax.legend(framealpha=0.9)
for bar in b1 + b2:
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.05, int(h),
                ha="center", va="bottom", fontweight="bold")
ax.set_ylim(0, max(max(counts_025), max(counts_05)) + 1.5)
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
fig.tight_layout()
fig.savefig("fig4_fingerprinting.png", dpi=150)
plt.show()

In [ ]:
# Cell 8: Fig 5 - C-value heatmap
n_layers = 8
proj_order = ["q_proj", "k_proj", "v_proj", "o_proj",
              "gate_proj", "up_proj", "down_proj"]
proj_labels = ["Q", "K", "V", "O", "Gate", "Up", "Down"]

c_matrix = np.full((n_layers, len(proj_order)), np.nan)
for key, val in c_vals.items():
    for li in range(n_layers):
        if f"layer.{li}." in key:
            for pi, pat in enumerate(proj_order):
                if pat in key:
                    c_matrix[li, pi] = val; break
            break

fig, ax = plt.subplots(figsize=(9, 5))
cmap = sns.diverging_palette(250, 15, s=75, l=45, n=256, center="light")
sns.heatmap(c_matrix, ax=ax, annot=True, fmt=".4f", cmap=cmap,
            xticklabels=proj_labels,
            yticklabels=[f"Layer {i}" for i in range(n_layers)],
            vmin=0.15, vmax=0.60, linewidths=1, linecolor="white",
            cbar_kws={"label": "c value", "shrink": 0.8})
ax.set_title("Learned c values per layer and projection")
ax.set_xlabel("Projection type")
ax.set_ylabel("Layer depth")
fig.tight_layout()
fig.savefig("fig5_c_heatmap.png", dpi=150)
plt.show()

In [ ]:
# Cell 9: Fig 6 - Snapping loss landscape
df_lc = logs["learned_c"]
if "total_loss" in df_lc.columns:
    fig, ax1 = plt.subplots(figsize=(10, 5))
    l1 = ax1.plot(df_lc["step"], df_lc["task_loss"], color="#4C72B0",
                  linewidth=2, label="Task loss (cross-entropy)")
    l2 = ax1.plot(df_lc["step"], df_lc["total_loss"], color="#C44E52",
                  linewidth=2, linestyle="--", label="Total loss (task + l*penalty)")
    ax2 = ax1.twinx()
    l3 = ax2.plot(df_lc["step"], df_lc["lambda"], color="#55A868",
                  linewidth=2, linestyle=":", label="l(t)")
    ax2.set_ylabel("l(t)", color="#55A868")
    ax2.tick_params(axis="y", labelcolor="#55A868")
    
    snap_idx = (df_lc["lambda"] > 0).idxmax() if (df_lc["lambda"] > 0).any() else None
    if snap_idx is not None:
        ax1.axvline(x=df_lc.loc[snap_idx, "step"], color="gray", linestyle="--", alpha=0.6)
    
    ax1.set_xlabel("Step"); ax1.set_ylabel("Loss")
    ax1.set_title("Loss landscape during the snapping phase")
    ax1.legend(l1 + l2 + [plt.Line2D([0],[0], color="gray", linestyle="--")],
               ["Task loss", "Total loss", "Snap start"],
               loc="upper left", framealpha=0.9)
    fig.tight_layout()
    fig.savefig("fig6_snapping_landscape.png", dpi=150)
    plt.show()

In [ ]:
# Cell 10: Fig 7 - Snapping convergence over time
if "n_snapped_025" in df_lc.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(df_lc["step"], df_lc["n_snapped_025"], color="#4C72B0",
            linewidth=2, marker="o", label="Snapped to 0.25")
    ax.plot(df_lc["step"], df_lc["n_snapped_05"], color="#DD8452",
            linewidth=2, marker="s", label="Snapped to 0.50")
    total_c = 56  # 8 layers * 7 projections
    ax.axhline(y=total_c, color="gray", linestyle=":", alpha=0.5, label=f"Total c params ({total_c})")
    
    snap_idx = (df_lc["lambda"] > 0).idxmax() if (df_lc["lambda"] > 0).any() else None
    if snap_idx is not None:
        ax.axvline(x=df_lc.loc[snap_idx, "step"], color="red", linestyle="--", alpha=0.4)
    
    ax.set_xlabel("Step"); ax.set_ylabel("Number snapped")
    ax.set_title("Snapping convergence over training steps")
    ax.legend(framealpha=0.9)
    ax.set_ylim(0, total_c * 1.15)
    ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    fig.tight_layout()
    fig.savefig("fig7_snapping_convergence.png", dpi=150)
    plt.show()

In [ ]:
# Cell 11: Fig 8 - C-value trajectories per layer
if "c_values" in df_lc.columns:
    proj_short = {"q_proj": "Q", "k_proj": "K", "v_proj": "V", "o_proj": "O",
                  "gate_proj": "G", "up_proj": "U", "down_proj": "D"}
    fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharex=True, sharey=True)
    axes = axes.flatten()
    colors_proj = plt.cm.Set2(np.linspace(0, 1, len(proj_short)))
    
    for layer_idx in range(8):
        ax = axes[layer_idx]
        for pi, (pat, plabel) in enumerate(proj_short.items()):
            key = f"layer.{layer_idx}.{pat}"
            traj, steps = [], []
            for _, row in df_lc.iterrows():
                cv = row["c_values"].get(key)
                if cv is not None:
                    traj.append(cv); steps.append(row["step"])
            if traj:
                ax.plot(steps, traj, color=colors_proj[pi], linewidth=1.5, label=plabel, alpha=0.8)
        ax.axhline(y=0.25, color="gray", linestyle=":", alpha=0.4)
        ax.axhline(y=0.50, color="gray", linestyle=":", alpha=0.4)
        ax.set_title(f"Layer {layer_idx}")
        if layer_idx >= 4: ax.set_xlabel("Step")
        if layer_idx % 4 == 0: ax.set_ylabel("c value")
        ax.set_ylim(0.15, 0.60)
    
    axes[-1].legend(loc="center", framealpha=0.9, ncol=1, bbox_to_anchor=(0.5, 0.5))
    fig.suptitle("C-value trajectories per layer during training", fontsize=14)
    fig.tight_layout()
    fig.savefig("fig8_c_trajectories.png", dpi=150)
    plt.show()

In [ ]:
# Cell 12: Fig 9 - Energy / MAC comparison
from benchmark import count_macs_per_token, QUANT_ENERGY

cfg_10m = dict(vocab_size=4096, hidden_dim=256, num_layers=8,
               num_heads=8, ffn_dim=1024, max_seq_len=512)
macs = count_macs_per_token(cfg_10m)

quant_baselines = ["full_precision", "bitnet", "fixed_c_025", "fixed_c_05"]
energy_labels = ["Full precision\n(FP32 ADD+MULT)", "BitNet b1.58\n(MUX/neg + ADD)",
                 "Quaternary c=0.25\n(SHIFT-2 + ADD)", "Quaternary c=0.5\n(SHIFT-1 + ADD)"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# MAC breakdown
per_layer_attn = (
    3 * cfg_10m["hidden_dim"] * cfg_10m["hidden_dim"]  # QKV proj
    + cfg_10m["num_heads"] * cfg_10m["max_seq_len"] * (cfg_10m["hidden_dim"] // cfg_10m["num_heads"]) * 2  # QK^T + attn@V
    + cfg_10m["hidden_dim"] * cfg_10m["hidden_dim"]  # O proj
)
per_layer_ffn = 3 * cfg_10m["hidden_dim"] * cfg_10m["ffn_dim"]  # gate + up + down

ax1.pie([per_layer_attn * cfg_10m["num_layers"], per_layer_ffn * cfg_10m["num_layers"]],
        labels=[f"Attention\n({per_layer_attn:,} MACs/layer)",
                f"FFN\n({per_layer_ffn:,} MACs/layer)"],
        autopct="%1.1f%%", colors=["#4C72B0", "#DD8452"],
        startangle=90, textprops={"fontsize": 10})
ax1.set_title("MAC distribution per token")

# Energy bar
pj_per_mac = [QUANT_ENERGY[b] for b in quant_baselines]
nJ_per_tok = [macs["total_macs"] * pj / 1000 for pj in pj_per_mac]
bars = ax2.bar(energy_labels, nJ_per_tok,
               color=[COLORS[0], COLORS[1], COLORS[4], COLORS[5]],
               width=0.5, edgecolor="white")
ax2.set_ylabel("Energy per token (nJ)")
ax2.set_title("Energy per token (Horowitz ISSCC 2014, 45nm)")
ax2.tick_params(axis="x", rotation=15)
for bar, val in zip(bars, nJ_per_tok):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f"{val:.1f}", ha="center", va="bottom", fontweight="bold")
fp_nJ = nJ_per_tok[0]
for bar, val in zip(bars, nJ_per_tok):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 0.6,
             f"{val/fp_nJ*100:.0f}% of FP", ha="center", va="center",
             fontsize=8, color="white", fontweight="bold")
fig.tight_layout()
fig.savefig("fig9_energy_comparison.png", dpi=150)
plt.show()

In [ ]:
# Cell 13: Summary table
print("\n" + "=" * 100)
print("FINAL SUMMARY: Tetranet 10M Model Comparison")
print("=" * 100)
print(f"\nSystem: {DEVICE} ({VRAM:.1f} GB) | "
      f"Model: 10M params (dim=256, layers=8, heads=8, ffn=1024)")
print(f"Dataset: 100K TinyStories (~3.3M tokens) | "
      f"Validation: 2,500 TinyStories (~80K tokens)")

header = f"\n{'Baseline':<22s} {'Best loss':>10s} {'PPL':>8s} {'DFP':>8s} " \
         f"{'MACs/tok':>10s} {'nJ/tok':>8s} {'vs FP':>8s}"
print(header); print("-" * 70)

for i, name in enumerate(BASELINE_NAMES):
    best_loss = logs[name]["task_loss"].min()
    ppl = math.exp(best_loss) if best_loss < 10 else float("nan")
    delta_ppl = ppl - ppl_results.get("full_precision", {}).get("ppl", 0)
    qb_map = {"full_precision": "full_precision", "bitnet": "bitnet",
              "uniform_2bit": "full_precision", "fixed_c_025": "fixed_c_025",
              "fixed_c_05": "fixed_c_05", "learned_c": "fixed_c_05"}
    qb = qb_map[name]
    nJ = macs["total_macs"] * QUANT_ENERGY.get(qb, 4.6) / 1000 if qb in QUANT_ENERGY else 0
    fp_nj = macs["total_macs"] * QUANT_ENERGY["full_precision"] / 1000
    print(f"{name:<22s} {best_loss:>10.4f} {ppl:>8.2f} {delta_ppl:>+8.2f} "
          f"{macs['total_macs']:>10,} {nJ:>8.1f} {nJ/fp_nj*100:>7.0f}%")

print(f"\n--- learned_c snapping ---")
c_vals = model_lc.get_c_values()
n_025 = sum(1 for v in c_vals.values() if abs(v - 0.25) < 0.02)
n_05 = sum(1 for v in c_vals.values() if abs(v - 0.50) < 0.02)
print(f"  Snapped to 0.25: {n_025}/{len(c_vals)} ({n_025/len(c_vals)*100:.1f}%)")
print(f"  Snapped to 0.50: {n_05}/{len(c_vals)} ({n_05/len(c_vals)*100:.1f}%)")
print(f"  Failed to snap:  {len(c_vals)-n_025-n_05}/{len(c_vals)} "
      f"({(len(c_vals)-n_025-n_05)/len(c_vals)*100:.1f}%)")
print("=" * 100)

In [ ]:
# Cell 14: Copy all outputs to /kaggle/working/
import glob
for fname in glob.glob("*.png") + glob.glob("*_final.pt") + glob.glob("*.csv"):
    shutil.copy(fname, "/kaggle/working/")
print("\nFinal outputs:")
for f in sorted(os.listdir("/kaggle/working/")):
    sz = os.path.getsize(f"/kaggle/working/{f}")
    print(f"  {f:40s} {sz/1024:>8.1f} KB")
print("\nDone! Download from Kaggle -> Data -> Output tab.")